# STAT 207 Homework 10 [25 points]

## Feature Selection for Linear Models

Due: Monday, April 27, end of day (11:59 pm CT)

Late submissions accepted until Tuesday, April 28 at noon

<hr>

## Imports 

Run the following code cell to import the necessary packages into the file.  You may import additional packages, as needed for this assignment.

In [12]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns; sns.set()
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
import statsmodels.formula.api as smf

## The Data

A famous study called "SUPPORT" (Study to Understand Prognoses Preferences Outcomes and Risks of Treatment) was conducted to determine what factors affected or predicted outcomes, including how long a patient remained in the hospital.

We will use a random sample of 580 seriously ill hospitalized patients from the SUPPORT study, with the following variables:

- **`Days`**: day to death or hospital discharge
- **`Age`**: age on day of hospital admission
- **`Sex`**: female or male
- **`Comorbidity`**: patient diagnosed with more than one chronic disease
- **`EdYears`**: years of education
- **`Education`**: education level: high or low
- **`Income`**: income level: high or low
- **`Charges`**: hospital charges, in dollars
- **`Care`**: level of care required: high or low
- **`Race`**: Non-white or white
- **`Pressure`**: Blood pressure, in mmHg
- **`Blood`**: white blood cell count, in gm/dL
- **`Rate`**: heart rate, in bpm

Run the code in the cell below to read in the cleaned data and separate the data into a training and test set for this document.  The data is saved as `df` with this code, and then separated into a `df_train` and `df_test` for later analysis.  

In [13]:
df = pd.read_csv('hospital.csv')
df_train, df_test = train_test_split(df, test_size = 0.20, random_state = 9876)

## 1. Research Purpose [0 points]

Question 1 does not require any analysis.

I'd suggest turning to Q1 on Gradescope here to answer the conceptual questions.

## 2. Fitting our Full Model [1 point]

For the purposes of this question, we want to be sure that we only include variables that could be determined or anticipated upon admittance to the hospital, so that our model can be applied to new patients.

**a)** Fit a linear model to the training data to predict the length of the hospital stay (**`Days`**) with the following predictor variables: `Age`, `Sex`, `Comorbidity`, `EdYears`, `Education`, `Income`, `Care`, `Race`, `Pressure`, `Blood`, and `Rate`.  Print a summary of this linear model.

In [14]:
full_model = smf.ols(
    formula='Days ~ Age + Sex + Comorbidity + EdYears + Education + Income + Care + Race + Pressure + Blood + Rate',
    data=df_train
).fit()

full_model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                   Days   R-squared:                       0.150
Model:                            OLS   Adj. R-squared:                  0.129
Method:                 Least Squares   F-statistic:                     7.230
Date:                Tue, 28 Apr 2026   Prob (F-statistic):           2.13e-11
Time:                        01:38:50   Log-Likelihood:                -2042.3
No. Observations:                 464   AIC:                             4109.
Df Residuals:                     452   BIC:                             4158.
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
======================================================================================
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             16.3417      9.863      1.657      0.098      -3.041      35.724
Sex[T.male]           -2.3662      1.912     -1.238      0.216      -6.124       1.391
Comorbidity[T.yes]    -2.1125      3.139     -0.673      0.501      -8.281       4.056
Education[T.low]       0.1577      2.969      0.053      0.958      -5.677       5.992
Income[T.low]         -1.5625      2.090     -0.748      0.455      -5.670       2.545
Care[T.low]          -11.5546      2.048     -5.641      0.000     -15.580      -7.529
Race[T.white]          3.4544      2.379      1.452      0.147      -1.222       8.131
Age                   -0.0800      0.063     -1.270      0.205      -0.204       0.044
EdYears               -0.3958      0.392     -1.011      0.313      -1.165       0.374
Pressure               0.1224      0.036      3.421      0.001       0.052       0.193
Blood                  0.0488      0.119      0.409      0.683      -0.186       0.283
Rate                   0.0703      0.032      2.196      0.029       0.007       0.133
==============================================================================
Omnibus:                      390.399   Durbin-Watson:                   1.981
Prob(Omnibus):                  0.000   Jarque-Bera (JB):             7914.321
Skew:                           3.615   Prob(JB):                         0.00
Kurtosis:                      21.897   Cond. No.                     1.60e+03
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 1.6e+03. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

**b)** I'd suggest turning to Q2 on Gradescope here.  

I don't anticipate a need to perform any more calculations or analyses for this problem.  This space is available for any optional calculations or analyses that you'd like.

## 3. Metrics for Model Fit [0 points]

I'd suggest turning to Q3 on Gradescope here to answer the conceptual questions.

Question 3 does not require any analysis.  This space is available for any optional calculations or analyses that you'd like.

## 4. Selecting a Parsimonious Model [3 points]

**a)** We'll perform model selection using backwards elimination starting from the full model fit in Question 2 and using the $R^2_{\text{adj}}$ as our metric using our training data.  Be sure to show your work.  That is, don't delete any models that you fit.  Add as many code cells below as you need.

In [19]:
current_model = full_model
predictors = [
    'Age', 'Sex', 'Comorbidity', 'EdYears', 'Education', 'Income',
    'Care', 'Race', 'Pressure', 'Blood', 'Rate'
]


In [20]:


while True:
    results = []
    for predictor in predictors:
        reduced_predictors = predictors.copy()
        reduced_predictors.remove(predictor)
        formula = 'Days ~ ' + ' + '.join(reduced_predictors)
        model = smf.ols(formula=formula, data=df_train).fit()
        results.append({
            'removed': predictor,
            'adj_r2': model.rsquared_adj
        })
    results_df = pd.DataFrame(results).sort_values('adj_r2', ascending=False)
    display(results_df)
    best_removed = results_df.iloc[0]['removed']
    best_adj_r2 = results_df.iloc[0]['adj_r2']
    current_adj_r2 = current_model.rsquared_adj
    print("Current adjusted R^2:", current_adj_r2)
    print("Best adjusted R^2 after removal:", best_adj_r2)
    print("Best variable to remove:", best_removed)
    if best_adj_r2 > current_adj_r2:
        predictors.remove(best_removed)
        current_formula = 'Days ~ ' + ' + '.join(predictors)
        current_model = smf.ols(
            formula=current_formula,
            data=df_train
        ).fit()
        print("Removing:", best_removed)
        print("New predictors:", predictors)
    else:
        break

,removed,adj_r2
4,Education,0.130855
9,Blood,0.130539
2,Comorbidity,0.129989
5,Income,0.129786
3,EdYears,0.128896
1,Sex,0.127915
0,Age,0.127760
7,Race,0.126808
10,Rate,0.121591
8,Pressure,0.108359


Current adjusted R^2: 0.1289374762917086
Best adjusted R^2 after removal: 0.13085492642170338
Best variable to remove: Education
Removing: Education
New predictors: ['Age', 'Sex', 'Comorbidity', 'EdYears', 'Income', 'Care', 'Race', 'Pressure', 'Blood', 'Rate']


,removed,adj_r2
8,Blood,0.132451
2,Comorbidity,0.131903
4,Income,0.131700
1,Sex,0.129827
0,Age,0.129622
3,EdYears,0.128761
6,Race,0.128650
9,Rate,0.123511
7,Pressure,0.110305
5,Care,0.071690


Current adjusted R^2: 0.13085492642170338
Best adjusted R^2 after removal: 0.13245059689240268
Best variable to remove: Blood
Removing: Blood
New predictors: ['Age', 'Sex', 'Comorbidity', 'EdYears', 'Income', 'Care', 'Race', 'Pressure', 'Rate']


,removed,adj_r2
4,Income,0.133348
2,Comorbidity,0.133304
0,Age,0.131362
1,Sex,0.131200
3,EdYears,0.130214
6,Race,0.130120
8,Rate,0.124639
7,Pressure,0.111641
5,Care,0.072079


Current adjusted R^2: 0.13245059689240268
Best adjusted R^2 after removal: 0.13334784894852936
Best variable to remove: Income
Removing: Income
New predictors: ['Age', 'Sex', 'Comorbidity', 'EdYears', 'Care', 'Race', 'Pressure', 'Rate']


,removed,adj_r2
2,Comorbidity,0.134026
1,Sex,0.132463
0,Age,0.132283
3,EdYears,0.132066
5,Race,0.130483
7,Rate,0.125791
6,Pressure,0.112673
4,Care,0.072755


Current adjusted R^2: 0.13334784894852936
Best adjusted R^2 after removal: 0.13402570920886248
Best variable to remove: Comorbidity
Removing: Comorbidity
New predictors: ['Age', 'Sex', 'EdYears', 'Care', 'Race', 'Pressure', 'Rate']


,removed,adj_r2
2,EdYears,0.133022
1,Sex,0.132916
0,Age,0.132766
4,Race,0.131170
6,Rate,0.126782
5,Pressure,0.113243
3,Care,0.064961


Current adjusted R^2: 0.13402570920886248
Best adjusted R^2 after removal: 0.13302234791793477
Best variable to remove: EdYears


In [22]:
selected_model = smf.ols(
    formula='Days ~ EdYears + Sex + Age + Race + Rate + Pressure + Care',
    data=df_train
).fit()


**b)** I'd suggest turning to Q4 on Gradescope here.  

I don't anticipate a need to perform any more calculations or analyses for this problem.  This space is available for any optional calculations or analyses that you'd like.

## 5. Measuring our Test Data [0 points] 

For this question, you'll calculate an appropriate measure of model fit through an $R^2$ (either unadjusted or adjusted $R^2$) on our test data based on our model selected and fit to our training data in Question 4.  You do not need to fit a new model for this question.  

You will need to perform additional calculations or analyses for this problem.  This space is available for those calculations, although there are no points associated with the code.  You will enter the results of your calculation on Q5 on Gradescope.

In [23]:
y_test = df_test['Days']
y_pred = selected_model.predict(df_test)

sse = ((y_test - y_pred) ** 2).sum()
sst = ((y_test - y_test.mean()) ** 2).sum()

test_r2 = 1 - sse / sst
test_r2


np.float64(0.06817312112215967)

## 6. AI Acknowledgement

Our course policy is that you should write all of your own interpretations and other narrative answers (phrases or sentences) yourself without the assistance of AI.  You may use AI to help guide your code, although you should write all of your own code yourself (not copy-paste from another source) and you should cite your use of AI.  I would encourage you to try to generate any necessary code yourself first using course resources and using AI as a debugging tool if/when you reach an error that you can't figure out or to help you perform any coding tasks that are more advanced than we've demonstrated during class (intended only for projects).  

Did you use AI on this assignment?  Did you use other resources outside of our course-provided resources on this assignment?

no

If you used AI or other resources, answer the following questions to cite your usage.

- Which AI and/or resources did you use (including links, if appropriate)?
- What prompts did you ask it?
- How did you integrate the responses into your assignment?  Specifically, which questions or parts are associated with this usage?

Note: answering these three questions are enough for our course but may not be enough for a different course or context.

n/a

Remember to keep all your cells and hit the save icon above periodically to checkpoint (save) your results on your local computer. Once you are satisified with your results restart the kernel and run all (Kernel -> Restart & Run All). **Make sure nothing has changed**. Checkpoint and exit (File -> Save and Checkpoint + File -> Close and Halt). Follow the instructions on the Homework 10 Canvas Assignment to submit your notebook to GitHub.